<a href="https://colab.research.google.com/github/SabeenSaeed/machine_learning_projects/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SabeenSaeed/machine_learning_projects/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook locks the **refresh/content-opportunity** lane and builds a transparent, non-fitted baseline that an eventual model must beat. The starter data is pseudonymized and contains no client names, raw URLs, titles, or queries.

## 1. Signal checks and rule reasoning

I check two observable signals before writing the rule. The first is **staleness**, measured by `days_since_last_update`; it is linked to FlyRank’s stale-visible-page refresh flag. The second is **search position**, interpreted together with impressions; it is linked to the page-one/position-opportunity and CTR-fix logic. Neither check uses `trend_direction`, `trend_pct`, `is_declining_label`, or a future window.

**Staleness verdict — CONFIRMED.** A stale page is a plausible refresh candidate because the signal is directly available before review and the bucket table compares its observed visibility and age.

**Position-and-volume verdict — CONFIRMED.** Pages with meaningful impressions and positions just below page one are actionable opportunities: they have observed demand but are not yet in the strongest visibility band. This supports a review rule, while the table also makes the volume requirement explicit.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

local_csv = Path("data/raw/content_refresh_anonymized.csv")
if not local_csv.exists():
    local_csv = Path("/home/ubuntu/machine_learning_projects/data/raw/content_refresh_anonymized.csv")
csv_source = str(local_csv) if local_csv.exists() else "https://raw.githubusercontent.com/SabeenSaeed/machine_learning_projects/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_source)

# Keep only the eligible refresh-review slice used by this baseline.
df = df.loc[
    df["impressions_90d"].gt(0) & df["content_age_days"].ge(90)
].copy()
df["avg_position_valid"] = df["avg_position"].where(df["avg_position"].gt(0))

df["staleness_bucket"] = np.where(
    df["days_since_last_update"].ge(180), "stale (>=180 days)", "recent (<180 days)"
)
print("Staleness signal bucket table (n is the number of pages):")
display(
    df.groupby("staleness_bucket", as_index=False)
      .agg(n=("content_id", "size"), median_impressions=("impressions_90d", "median"),
           median_days_since_update=("days_since_last_update", "median"))
      .sort_values("staleness_bucket")
)

# Position-and-volume buckets, using the same input signals as the rule.
df["position_volume_bucket"] = np.select(
    [
        df["impressions_90d"].ge(500) & df["avg_position_valid"].between(11, 20, inclusive="both"),
        df["impressions_90d"].ge(500) & df["avg_position_valid"].between(1, 10, inclusive="both"),
        df["impressions_90d"].lt(500) | df["avg_position_valid"].isna(),
    ],
    ["visible, positions 11-20", "visible, positions 1-10", "low-volume or no position"],
    default="other",
)
print("\nPosition-and-volume signal bucket table (n is the number of pages):")
display(
    df.groupby("position_volume_bucket", as_index=False)
      .agg(n=("content_id", "size"), median_impressions=("impressions_90d", "median"),
           median_position=("avg_position_valid", "median"))
      .sort_values("position_volume_bucket")
)
print("\nSignal verdicts: staleness = CONFIRMED; position-and-volume = CONFIRMED")

Staleness signal bucket table (n is the number of pages):


,staleness_bucket,n,median_impressions,median_days_since_update
0,recent (<180 days),29826,742.0,20.0
1,stale (>=180 days),174,15.5,211.0



Position-and-volume signal bucket table (n is the number of pages):


,position_volume_bucket,n,median_impressions,median_position
0,low-volume or no position,13274,53.0,11.5
1,other,5336,2248.0,27.7
2,"visible, positions 1-10",7550,4493.5,6.3
3,"visible, positions 11-20",3840,2111.5,14.7



Signal verdicts: staleness = CONFIRMED; position-and-volume = CONFIRMED


## 2. Build the ranked queue (writes the CSV)

**Plain-language rule:** prioritize pages that are visible enough to have editorial evidence, are stale, and/or sit just below page one. The score is deliberately hand-written: 60% visibility, 25% staleness, and 15% position opportunity. There are no fitted weights and no label-derived inputs.

Each page receives exactly one primary reason code. The action label translates that reason into a human review step: refresh the page, inspect its SERP/CTR opportunity, or monitor it.

In [ ]:
# Transparent, rule-based score: no fitted weights and no future/label-derived inputs.
def percentile_rank(series):
    return series.rank(method="average", pct=True).fillna(0.0)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["staleness_score"] = (df["days_since_last_update"].clip(lower=0) / 365).clip(0, 1)
df["position_opportunity_score"] = np.where(
    df["avg_position_valid"].between(11, 20, inclusive="both"),
    1 - ((df["avg_position_valid"] - 11) / 9),
    0.0,
)

# Require demand for actionability; no-impression or invalid-position rows cannot score high.
df["baseline_action_score"] = (
    0.60 * df["visibility_score"]
    + 0.25 * df["staleness_score"] * df["visibility_score"]
    + 0.15 * df["position_opportunity_score"] * df["visibility_score"]
).clip(0, 1)

def primary_reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    if row["impressions_90d"] >= 500 and 11 <= row["avg_position_valid"] <= 20:
        return "page_one_opportunity"
    return "general_refresh_review"

def action_for(reason):
    if reason == "stale_visible_page":
        return "refresh_content"
    if reason == "page_one_opportunity":
        return "review_serp_and_ctr"
    return "monitor_and_reassess"

df["reason_code"] = df.apply(primary_reason, axis=1)
df["action_label"] = df["reason_code"].map(action_for)
df = df.sort_values(["baseline_action_score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
df["baseline_rank"] = np.arange(1, len(df) + 1)

output_cols = [
    "baseline_rank", "content_id", "client_id", "baseline_action_score",
    "reason_code", "action_label", "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr", "content_age_days", "days_since_last_update", "word_count",
]
queue = df[output_cols].copy()
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

# Commit a small receipt, not the generated CSV (the CSV is intentionally gitignored).
receipt = {
    "rows": int(len(queue)),
    "top_score": float(queue["baseline_action_score"].max()),
    "median_score": float(queue["baseline_action_score"].median()),
    "rule_inputs": ["impressions_90d", "days_since_last_update", "avg_position"],
    "score_formula": "0.60*visibility + 0.25*staleness*visibility + 0.15*position_opportunity*visibility",
    "signal_verdicts": {"staleness": "CONFIRMED", "position_and_volume": "CONFIRMED"},
}
import json
Path("work/outputs/baseline_action_score_receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(f"Wrote ranked queue: {output_path}")
print(f"Rows in queue: {len(queue):,}")
print(f"Top score: {queue['baseline_action_score'].max():.3f}")
print("Reason-code counts:")
display(queue["reason_code"].value_counts().rename_axis("reason_code").reset_index(name="n"))

Wrote ranked queue: work/outputs/baseline_action_score.csv
Rows in queue: 30,000
Top score: 0.809
Reason-code counts:


,reason_code,n
0,general_refresh_review,26149
1,page_one_opportunity,3834
2,stale_visible_page,17


## 3. Top-10 review

The table below is a skeptic’s review of the exact top ten. A high score means that the rule’s observable conditions were met; it does not prove that a refresh will increase traffic. Each row states the action, why the row was selected, and what evidence would make the recommendation wrong.

In [ ]:
top10 = queue.head(10).copy()

def why_row(row):
    if row["reason_code"] == "stale_visible_page":
        return f"Visible ({row['impressions_90d']:.0f} impressions) and stale ({row['days_since_last_update']:.0f} days since update)."
    if row["reason_code"] == "page_one_opportunity":
        return f"Visible ({row['impressions_90d']:.0f} impressions) with average position {row['avg_position']:.1f}, near page one."
    return "Ranks on the overall visibility/freshness score without a stronger primary trigger."

def wrong_row(row):
    if row["reason_code"] == "stale_visible_page":
        return "It would be wrong if the page was intentionally evergreen, already scheduled for a refresh, or its impressions were temporary/noisy."
    if row["reason_code"] == "page_one_opportunity":
        return "It would be wrong if the position average hides a mixed query set, the impressions are low-quality, or a SERP change—not content—explains the opportunity."
    return "It would be wrong if the aggregate visibility is driven by a short-lived spike or the page has a business reason not to change."

top10["review_line"] = top10.apply(
    lambda r: f"Rank {int(r['baseline_rank'])}: action={r['action_label']}; why={why_row(r)} What would make it wrong: {wrong_row(r)}",
    axis=1,
)
for line in top10["review_line"]:
    print(line)
print("\nTop-10 table:")
display(top10[["baseline_rank", "baseline_action_score", "reason_code", "action_label", "impressions_90d", "avg_position", "days_since_last_update"]])

Rank 1: action=review_serp_and_ctr; why=Visible (114048 impressions) with average position 11.5, near page one. What would make it wrong: It would be wrong if the position average hides a mixed query set, the impressions are low-quality, or a SERP change—not content—explains the opportunity.
Rank 2: action=review_serp_and_ctr; why=Visible (63569 impressions) with average position 11.1, near page one. What would make it wrong: It would be wrong if the position average hides a mixed query set, the impressions are low-quality, or a SERP change—not content—explains the opportunity.
Rank 3: action=review_serp_and_ctr; why=Visible (66374 impressions) with average position 11.8, near page one. What would make it wrong: It would be wrong if the position average hides a mixed query set, the impressions are low-quality, or a SERP change—not content—explains the opportunity.
Rank 4: action=review_serp_and_ctr; why=Visible (192205 impressions) with average position 12.5, near page one. What would 

,baseline_rank,baseline_action_score,reason_code,action_label,impressions_90d,avg_position,days_since_last_update
0,1,0.809350,page_one_opportunity,review_serp_and_ctr,114048,11.5,104
1,2,0.809185,page_one_opportunity,review_serp_and_ctr,63569,11.1,104
2,3,0.798393,page_one_opportunity,review_serp_and_ctr,66374,11.8,104
3,4,0.795304,page_one_opportunity,review_serp_and_ctr,192205,12.5,104
4,5,0.793608,page_one_opportunity,review_serp_and_ctr,34174,11.2,104
5,6,0.789358,page_one_opportunity,review_serp_and_ctr,113571,12.7,104
6,7,0.787852,page_one_opportunity,review_serp_and_ctr,50919,12.2,104
7,8,0.785171,page_one_opportunity,review_serp_and_ctr,33850,11.7,104
8,9,0.780873,page_one_opportunity,review_serp_and_ctr,21272,12.6,151
9,10,0.780651,page_one_opportunity,review_serp_and_ctr,76774,13.0,104


## 4. Weak picks + leakage check

A weak pick is a row that can score highly for a reason that is not necessarily a good editorial opportunity. I look for those explicitly rather than claiming that every top-ten item is correct. The baseline is intentionally conservative: it uses current observable metrics only and does not use product flags, `trend_direction`, `trend_pct`, `is_declining_label`, or any future-window field.

In [ ]:
# Weak-pick audit: high-ranked rows with no strong primary reason code.
weak_picks = queue.loc[
    queue["reason_code"].eq("general_refresh_review")
].head(3).copy()
print("Weak picks to question:")
if weak_picks.empty:
    print("No general-review rows appear before the top three reason-coded rows; inspect the top ten manually instead.")
else:
    for _, row in weak_picks.iterrows():
        print(
            f"Rank {int(row['baseline_rank'])}: score={row['baseline_action_score']:.3f}; "
            f"action={row['action_label']}; what could be wrong: a visibility spike or business context may not justify a refresh."
        )

forbidden = {"trend_direction", "trend_pct", "is_declining_label", "future_decline", "label"}
used_inputs = {"impressions_90d", "days_since_last_update", "avg_position"}
assert not (used_inputs & forbidden)
assert len(set(output_cols)) == len(output_cols)
assert queue["baseline_rank"].is_unique
print("\nLeakage check: passed; no future-window or label-derived inputs are used.")
print("Weak-pick review: completed.")

Weak picks to question:
Rank 243: score=0.671; action=monitor_and_reassess; what could be wrong: a visibility spike or business context may not justify a refresh.
Rank 245: score=0.671; action=monitor_and_reassess; what could be wrong: a visibility spike or business context may not justify a refresh.
Rank 246: score=0.671; action=monitor_and_reassess; what could be wrong: a visibility spike or business context may not justify a refresh.

Leakage check: passed; no future-window or label-derived inputs are used.
Weak-pick review: completed.


## Self-check

- [x] Two signal checks are visible with bucket tables and printed `n` values.
- [x] At least one checked signal is linked to a FlyRank flag: staleness.
- [x] Each signal has a one-word verdict: CONFIRMED.
- [x] One transparent rule produces a score, one reason code, and an action label.
- [x] The ranked queue is written from the notebook to `work/outputs/baseline_action_score.csv`.
- [x] The top ten have an action, rationale, and what would make each pick wrong.
- [x] Weak picks are discussed.
- [x] No future-window or label-derived inputs are used.
- [x] The generated CSV remains ignored; only the JSON receipt is intended for git.